### Main User Interface

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Config
ENV_PATH = r"B:\3. Prog\2. Projects\7. Logistics and supply chain\B. Main AI Agent\LLM & ENV\LLM_API.env"
DB_PATH = r"B:\3. Prog\2. Projects\7. Logistics and supply chain\B. Main AI Agent\data\logistics.db"

load_dotenv(dotenv_path=ENV_PATH, override=True)

def get_llm():
    return ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

def get_db_connection():
    return duckdb.connect(DB_PATH)

def get_schema_info():
    conn = get_db_connection()
    try:
        tables = conn.execute("SHOW TABLES").fetchall()
        schema = ""
        for (table_name,) in tables:
            schema += f"\nTable: {table_name}\n"
            cols = conn.execute(f"DESCRIBE {table_name}").fetchall()
            for col in cols:
                schema += f" - {col[0]} ({col[1]})\n"
        return schema
    finally:
        conn.close()

c:\Users\Pritam\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### SQL_KPI_&_EDA_Redflags_Agent

In [ ]:
# Prototype (Working)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px  # Added for interactive charts
import plotly.graph_objects as go
from bokeh.plotting import figure, show, output_notebook

# Initialize Bokeh to work inside the Notebook
output_notebook()

def get_autonomous_context():
    conn = get_db_connection()
    tables = conn.execute("SHOW TABLES").fetchall()
    context = "DATABASE DICTIONARY & SEMANTIC MAP:\n"
    for (t_name,) in tables:
        cols = conn.execute(f"DESCRIBE {t_name}").fetchdf()
        context += f"\n--- TABLE: {t_name} ---\n"
        for _, row in cols.iterrows():
            col_name = row['column_name']
            col_type = row['column_type']
            stats = ""
            if "DOUBLE" in col_type or "INTEGER" in col_type:
                res = conn.execute(f"SELECT MIN({col_name}), MAX({col_name}) FROM {t_name}").fetchone()
                if res[0] is not None:
                    stats = f" | Range: {round(res[0],2)} to {round(res[1],2)}"
            context += f"Column: {col_name} ({col_type}){stats}\n"
    conn.close()
    return context

def run_logistics_agent(user_query, mode='KPI'):
    brain = get_llm()
    db_context = get_autonomous_context()
    
    if mode == 'KPI':
        prompt = f"{db_context}\nTASK: Generate DuckDB SQL for: {user_query}. Return ONLY SQL."
    else:
        prompt = f"""
        {db_context}
        TASK: Write Python code to analyze: {user_query}
        Dataframe 'df' is pre-loaded.
        
        AVAILABLE LIBRARIES: 
        - px (Plotly Express) for interactive web charts.
        - sns/plt for static statistical charts.
        - bokeh for high-performance interactive visuals.
        
        RULES:
        1. If the user asks for 'interactive' or 'hover', use Plotly (px).
        2. Otherwise, default to Seaborn for speed.
        3. Return ONLY the Python code.
        """

    response = brain.invoke(prompt)
    content = response.content[0]['text'] if isinstance(response.content, list) else response.content
    content = content.replace("```python", "").replace("```sql", "").replace("```", "").strip()

    conn = get_db_connection()
    try:
        if mode == 'KPI':
            df = conn.execute(content).fetchdf()
            analysis = brain.invoke(f"Summarize these results: {df.head(5).to_string()}").content
            return df, analysis
        else:
            # Join dimensions for EDA
            df = conn.execute("""
                SELECT f.*, d.driver_behavior_score, d.fatigue_monitoring_score, v.fuel_consumption_rate, r.route_risk_level
                FROM fact_shipments f
                LEFT JOIN dim_drivers d ON f.driver_id = d.driver_id
                LEFT JOIN dim_vehicles v ON f.vehicle_id = v.vehicle_id
                LEFT JOIN dim_routes r ON f.route_id = r.route_id
                LIMIT 30000
            """).fetchdf()
            
            # Injecting all libraries into the execution sandbox
            exec_globals = {
                "df": df, "plt": plt, "sns": sns, "pd": pd, 
                "np": np, "px": px, "go": go, "show": show, "figure": figure
            }
            exec(content, exec_globals)
            return "Visual Rendered", "Interactive Plot Complete"
    except Exception as e:
        return f"❌ Error: {e}", f"Failed code: {content}"
    finally:
        conn.close()

Loading BokehJS ...

The Architect scans the DB the very first time the user asks a question and saves it to _CACHED_ESSENCE. The Architect scans it once and remembers it. The Specialist uses that memory to be smart. This leaves more of the "Quota" available for solving actual logistics problems rather than describing the table structure.

In [31]:
# Cached Omni-Script (Working)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px 
import plotly.graph_objects as go
from bokeh.plotting import figure, show, output_notebook

output_notebook()

# GLOBAL CACHE: To save your API Quota
_CACHED_ESSENCE = None

def get_autonomous_context(force_refresh=False):
    global _CACHED_ESSENCE
    if _CACHED_ESSENCE and not force_refresh:
        return _CACHED_ESSENCE
    
    print("🔍 Architect mapping DB Essence (Scan only happens once)...")
    conn = get_db_connection()
    tables = conn.execute("SHOW TABLES").fetchall()
    context = "DATABASE DICTIONARY & SEMANTIC MAP:\n"
    for (t_name,) in tables:
        cols = conn.execute(f"DESCRIBE {t_name}").fetchdf()
        context += f"\n--- TABLE: {t_name} ---\n"
        for _, row in cols.iterrows():
            col_name = row['column_name']
            col_type = row['column_type']
            stats = ""
            if "DOUBLE" in col_type or "INTEGER" in col_type:
                # Optimized range check
                res = conn.execute(f"SELECT MIN({col_name}), MAX({col_name}) FROM {t_name}").fetchone()
                if res[0] is not None:
                    stats = f" | Range: {round(res[0],2)} to {round(res[1],2)}"
            context += f"Column: {col_name} ({col_type}){stats}\n"
    conn.close()
    _CACHED_ESSENCE = context
    return _CACHED_ESSENCE

def run_logistics_agent(user_query, mode='KPI', provide_summary=True):
    brain = get_llm()
    db_context = get_autonomous_context()
    
    if mode == 'KPI':
        prompt = f"{db_context}\nTASK: Generate DuckDB SQL for: {user_query}. Use JOINs. 1.0=Delivered. Return ONLY SQL."
    else:
        prompt = f"{db_context}\nTASK: Write Python (px/sns) for: {user_query}. Dataframe 'df' is pre-loaded. Return ONLY code."

    # API CALL 1: Logic Generation
    response = brain.invoke(prompt)
    content = response.content[0]['text'] if isinstance(response.content, list) else response.content
    content = content.replace("```python", "").replace("```sql", "").replace("```", "").strip()

    conn = get_db_connection()
    try:
        if mode == 'KPI':
            df = conn.execute(content).fetchdf()
            
            # API CALL 2: Summary (Only if requested to save quota)
            analysis = "Summary disabled to save quota."
            if provide_summary:
                try:
                    analysis = brain.invoke(f"Summarize in 2 bullets: {df.head(5).to_string()}").content
                except:
                    analysis = "Quota exceeded for summary."
            return df, analysis
        else:
            df = conn.execute("""
                SELECT f.*, d.driver_behavior_score, d.fatigue_monitoring_score, v.fuel_consumption_rate, r.route_risk_level
                FROM fact_shipments f
                LEFT JOIN dim_drivers d ON f.driver_id = d.driver_id
                LEFT JOIN dim_vehicles v ON f.vehicle_id = v.vehicle_id
                LEFT JOIN dim_routes r ON f.route_id = r.route_id
                LIMIT 30000
            """).fetchdf()
            
            exec_globals = {"df": df, "plt": plt, "sns": sns, "pd": pd, "np": np, "px": px, "go": go, "show": show, "figure": figure}
            exec(content, exec_globals)
            return "Visual Rendered", "Success"
    except Exception as e:
        return f"❌ Error: {e}", content
    finally:
        conn.close()

Loading BokehJS ...

### Interacting with Main Agent:

In [ ]:
# The Interaction Loop Prototype

print("🚀 OMNI-AGENT READY (5.0M Rows Connected)")
user_input = input("Ask anything (e.g., 'What is the avg cost?' or 'Plot driver risk vs fatigue'): ")

# Routing Logic
eda_keywords = ['plot', 'graph', 'visual', 'chart', 'distribution', 'correlation', 'show me a map']

if any(word in user_input.lower() for word in eda_keywords):
    _, _ = run_logistics_agent(user_input, mode='EDA')
else:
    data, summary = run_logistics_agent(user_input, mode='KPI')
    display(data)
    # Handle both string and list responses from the LLM
    final_text = summary[0]['text'] if isinstance(summary, list) else summary
    print(f"\n💡 AI INSIGHT:\n{final_text}")

🚀 OMNI-AGENT READY (5.0M Rows Connected)


"❌ Error: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\\nPlease retry in 47.828714093s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDime


💡 AI INSIGHT:
Failed code: SELECT
  f.supplier_id,
  AVG(f.iot_temperature) AS average_temperature
FROM fact_shipments f
JOIN dim_vehicles v
  ON f.vehicle_id = v.vehicle_id
GROUP BY f.supplier_id;


The Architect scans the DB the very first time the user asks a question and saves it to _CACHED_ESSENCE. The Architect scans it once and remembers it. The Specialist uses that memory to be smart. This leaves more of the "Quota" available for solving actual logistics problems rather than describing the table structure.

In [32]:
# # The Interaction Loop (Cached)

print("🚀 OMNI-AGENT READY (Quota Optimized)")
user_input = input("Query: ")

# Smart Routing
is_eda = any(w in user_input.lower() for w in ['plot', 'graph', 'chart', 'visual'])
# Only ask for a summary if the user asks "why" or "summarize"
needs_summary = any(w in user_input.lower() for w in ['why', 'summarize', 'insight', 'explain'])

if is_eda:
    _, _ = run_logistics_agent(user_input, mode='EDA')
else:
    data, summary = run_logistics_agent(user_input, mode='KPI', provide_summary=needs_summary)
    display(data)
    if needs_summary:
        print(f"\n💡 AI INSIGHT:\n{summary}")



🚀 OMNI-AGENT READY (Quota Optimized)
🔍 Architect mapping DB Essence (Scan only happens once)...


ChatGoogleGenerativeAIError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 6.661317487s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT